# Lawsuits against towns: Gateway Cities, housing court, and trend
**Project:** GBH MassCourts Municipal Lawsuits

Builds on my DLS notebook -  using Reetom's cleaned court + finance file.

**Questions**
1. Is the decline in lawsuits against towns since 2006 real? (Step A done; Step B pending)
2. Gateway Cities vs. similar-size towns (35k–250k) and named towns — client request from Oct 16
3. What drives the gap? (housing court, which cities, city's role)

*Later:* town size and spending vs. lawsuits.

## 1. Load tools and data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/projectnb/sparkgrp/ds-wgbh-municipal-lawsuits-grp/ds-wgbh-municipal-lawsuits/fa26/data/analysis'
print('Folder found:', os.path.isdir(DATA))
print(os.listdir(DATA))

In [ ]:
df = pd.read_csv(f'{DATA}/municipal_litigation_finances_2000_2025.csv', dtype={'dor_code': str})
print(df.shape)
print(df.columns.tolist())

**Main file:** one row per town per year (351 towns × 26 years = 9,126 rows).
Has case counts (plaintiff, defendant, case types), budget, population, and rates per 10k residents.

## 2. Study window and town groups
- Keep **FY2006–2024** only: court records before 2006 are incomplete, and 2025 has no population.
- Make sure money and count columns are numbers.
- Flag the **26 Gateway Cities**.
- Use the **same size groups as slides 7–9**, so this connects to my DLS work.

In [ ]:
# 1) Keep the reliable years
d = df[df['year'].between(2006, 2024)].copy()

# 2) Make sure money and count columns are numbers (removes commas or $ if any)
num_cols = [c for c in d.columns if c.startswith(('revenue_', 'expenditure_', 'cases_'))] + ['population']
for c in num_cols:
    d[c] = pd.to_numeric(d[c].astype(str).str.replace(r'[,$]', '', regex=True), errors='coerce')

# 3) Gateway City flag
GATEWAY = ['Attleboro','Barnstable','Brockton','Chelsea','Chicopee','Everett','Fall River','Fitchburg',
           'Haverhill','Holyoke','Lawrence','Leominster','Lowell','Lynn','Malden','Methuen','New Bedford',
           'Peabody','Pittsfield','Quincy','Revere','Salem','Springfield','Taunton','Westfield','Worcester']
d['is_gateway'] = d['municipality'].isin(GATEWAY)

# 4) Size groups (same as slides 7–9)
bins   = [0, 5_000, 20_000, 50_000, np.inf]
labels = ['1. Under 5k', '2. 5k–20k', '3. 20k–50k', '4. 50k+']
d['size_band'] = pd.cut(d['population'], bins=bins, labels=labels)

# 5) Quick checks
print('Rows:', len(d), '| Years:', d.year.min(), '-', d.year.max())
print('Gateway Cities matched:', d.loc[d.is_gateway, 'municipality'].nunique(), 'of 26')
print('Towns per size group (2024):')
print(d[d.year == 2024].groupby('size_band', observed=True)['dor_code'].nunique())

## 3. Is the decline real?
Lawsuits against towns appear to fall by about half since 2006 (Natalie's chart).
Before GBH reports that, check it against **all civil cases filed** in MassCourtsPlus each year.
- If suits against towns fall **faster** than all civil filings → the decline is likely real.
- If both fall **together** → it may just be the database covering fewer cases.

**Step A:** yearly totals of suits against towns (this cell).
**Step B:** add all-civil-case totals from the MassCourtsPlus website (next session).

In [ ]:
yearly = (d[d['has_litigation_data'] == 1]
            .groupby('year')
            .agg(sued_cases=('cases_defendant', 'sum'),
                 all_town_cases=('cases_total', 'sum'),
                 population=('population', 'sum')))
yearly['sued_per_10k'] = (yearly['sued_cases'] / yearly['population'] * 1e4).round(2)
yearly

## 4. Gateway Cities vs. similar-size towns
Comparing Gateway Cities with **similar-size towns (35k–250k)**, and named
Somerville, Cambridge, Brookline (wealthier) and Weymouth, Randolph (near-Gateway).
Window: FY2015–2024 (recent 10 years).Rates are pooled per year: total cases ÷ total yearly residents.

In [ ]:
r = d[d.year.between(2015, 2024) & (d.has_litigation_data == 1)].copy()

# Comparison group: non-Gateway towns with 35k–250k residents in 2024
pop24 = d[d.year == 2024].set_index('municipality')['population']
similar = pop24[(pop24 >= 35_000) & (pop24 <= 250_000)].index
r['group'] = np.select([r.is_gateway, r.municipality.isin(similar)],
                       ['Gateway Cities', 'Similar-size (35k–250k)'], default='Other')

def rates(g):
    return pd.Series({'towns': g.dor_code.nunique(),
                      'sued_per_10k':      g.cases_defendant.sum() / g.population.sum() * 1e4,
                      'tax_lien_per_10k':  g.cases_tax_lien.sum()  / g.population.sum() * 1e4,
                      'town_suing_per_10k': g.cases_plaintiff.sum() / g.population.sum() * 1e4})

print('Group comparison, FY2015–2024')
display(r.groupby('group').apply(rates).round(2))

NAMED = ['Somerville', 'Cambridge', 'Brookline', 'Weymouth', 'Randolph']
print('Not found (check spelling):', [t for t in NAMED if t not in set(r.municipality)])
display(r[r.municipality.isin(NAMED)].groupby('municipality').apply(rates).round(2))

### Preliminary findings — Gateway vs. similar-size towns (FY2015–2024, per 10,000 residents per year)

| Group | Towns | Sued | Tax lien | Town suing |
|---|---|---|---|---|
| Gateway Cities | 26 | 1.57 | 1.82 | 7.55 |
| Similar-size (35k–250k) | 22 | 1.35 | 1.11 | 1.92 |
| All other towns | 303 | 1.61 | 2.27 | 2.91 |

1. **Gateway Cities are not sued much more than similar-size towns** (1.57 vs. 1.35, about 16% higher).
2. **Gateway Cities bring more cases as plaintiff** (7.55 vs. 1.92 pooled). The difference is in *suing*, not *being sued*.
   **Update:** the pooled 4× is pulled up by a few cities; the typical Gateway City is about **2.3×** (see robustness checks below).
3. **Tax liens explain only part of the gap** (1.82 vs. 1.11). Most of it comes from **housing court** cases (see Section 5).
4. **Named towns (client request):**
   - Cambridge (2.22) and Somerville (2.68) are sued *more* than the Gateway average — wealthier does not mean sued less.
   - Wealthier cities file almost no tax-lien cases (Cambridge 0.00, Somerville 0.11, Brookline 0.97 vs. Gateway 1.82). Cambridge's 0.00 may be a matching gap.
   - Weymouth and Randolph are low on every measure; "near-Gateway" towns do not look like Gateway Cities in court.

*Patterns, not causes. Counts depend on MassCourtsPlus coverage and Reetom's entity matching. Boston (250k+) is in "All other towns."*

In [ ]:
# A) Is the 'town suing' gap broad, or driven by a few cities?
city = (r[r.group != 'Other'].groupby(['group', 'municipality'])
          .apply(lambda g: g.cases_plaintiff.sum() / g.population.sum() * 1e4)
          .rename('town_suing_per_10k').reset_index())
print(city.groupby('group')['town_suing_per_10k'].describe().round(2))
peer_median = city.loc[city.group != 'Gateway Cities', 'town_suing_per_10k'].median()
gw = city[city.group == 'Gateway Cities']
print(f'\nGateway Cities above the similar-size median ({peer_median:.2f}):',
      (gw.town_suing_per_10k > peer_median).sum(), 'of', len(gw))

# B) What kinds of cases? (housing court, tax lien) per 10k
def mix(g):
    p = g.population.sum()
    return pd.Series({'town_suing': g.cases_plaintiff.sum()/p*1e4,
                      'tax_lien':   g.cases_tax_lien.sum()/p*1e4,
                      'housing':    g.cases_housing.sum()/p*1e4})
display(r[r.group != 'Other'].groupby('group').apply(mix).round(2))

- **Overall "town suing" gap is broad but modest:** 23 of 26 Gateway Cities are above the similar-size median (typical Gateway City 2.55 vs. 1.11 — about **2.3×**). The pooled 7.55 is pulled up by a few very high cities (max 27.5).
- **What drives it:** housing court cases — 5.50 vs. 0.09 per 10k per year. Tax liens are a small part.
- **The large housing court gap is concentrated** in a few cities (see Section 5).
- **Caveat:** Housing Court expanded statewide around 2017; checked with FY2018–2024 only (next cell).

In [ ]:
late = r[r.year.between(2018, 2024) & (r.group != 'Other')]
display(late.groupby('group').apply(mix).round(2))

### Result: the gap holds after Housing Court went statewide
FY2018–2024, per 10,000 residents per year:

| Group | Town suing | Tax lien | Housing court |
|---|---|---|---|
| Gateway Cities | 6.05 | 1.35 | 4.48 |
| Similar-size (35k–250k) | 1.83 | 0.97 | 0.10 |

- The housing court gap stays large after the 2017 expansion → **not explained by court coverage**.
- **Takeaway:** Gateway Cities are sued at about the same rate as similar-size towns. The big difference is housing court cases brought by the city — driven mainly by a few cities (Section 5).
- **Lead for GBH (to report out, not a conclusion):** are these cities using housing court to enforce health/building codes on landlords?

## 5. What is behind the housing court gap? (FY2018–2024)
A) Which Gateway Cities have the most housing court cases per 10k residents?

B) In those cases, is the city the plaintiff (bringing the case) or the defendant?

In [ ]:
# A) Top Gateway Cities by housing court cases per 10k (2018–2024)
gw_late = late[late.group == 'Gateway Cities']
top_housing = (gw_late.groupby('municipality')
                 .apply(lambda g: g.cases_housing.sum() / g.population.sum() * 1e4)
                 .sort_values(ascending=False).round(2))
print('Housing court cases per 10k, Gateway Cities (FY2018–2024):')
print(top_housing.head(10))
print('\nGateway Cities with fewer than 1 per 10k:', (top_housing < 1).sum(), 'of 26')

# B) City's role in housing court cases (counts only)
cases = pd.read_csv(f'{DATA}/municipal_cases.csv', dtype={'dor_code': str},
                    usecols=['municipality', 'case_type', 'file_date', 'has_plaintiff', 'has_defendant'])
cases['file_date'] = pd.to_datetime(cases['file_date'], errors='coerce')
cases['fy'] = cases.file_date.dt.year + (cases.file_date.dt.month >= 7).astype(int)
h = cases[cases.case_type.fillna('').str.lower().str.contains('housing court')
          & cases.fy.between(2018, 2024)
          & cases.municipality.isin(GATEWAY)].copy()
h['role'] = np.select([(h.has_plaintiff == 1) & (h.has_defendant == 0),
                       (h.has_defendant == 1) & (h.has_plaintiff == 0),
                       (h.has_plaintiff == 1) & (h.has_defendant == 1)],
                      ['City is plaintiff', 'City is defendant', 'Both'], default='Other / unknown')
print('\nGateway housing court cases, FY2018–2024:', len(h))
print((h.role.value_counts(normalize=True) * 100).round(1).astype(str) + '%')
print('\nHousing case types (top 5, share %):')
print((h.case_type.value_counts(normalize=True).head(5) * 100).round(1))

### Result: the housing court gap is concentrated, not broad (FY2018-2024)
- **Worcester (23.1 per 10k) and Springfield (14.7)** lead by far, then Chicopee, Holyoke, Leominster.
- **18 of 26** Gateway Cities have fewer than 1 housing court case per 10k - similar to comparison towns.
- The **city is the plaintiff in 99.3%** of Gateway housing court cases.
- **98.6% are "Housing Court Civil"** (evictions/"Summary Process" only 0.7%) - consistent with
  **code enforcement** against property owners (to be confirmed).
- **Corrected takeaway:** the Gateway "town suing" gap is driven mainly by a few cities,
  led by Worcester and Springfield - not by Gateway Cities as a group.

In [ ]:
OUT = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUT, exist_ok=True)

city_h = top_housing.sort_values()                     # all 26 Gateway Cities
peer_h = (late[late.group == 'Similar-size (35k–250k)'].cases_housing.sum()
          / late[late.group == 'Similar-size (35k–250k)'].population.sum() * 1e4)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#AD40D9' if v >= 1 else '#9A9A9A' for v in city_h]
ax.barh(city_h.index, city_h.values, color=colors)
ax.axvline(peer_h, color='#4F1C59', linestyle='--', linewidth=1)
ax.text(peer_h + 0.3, 0.3, f'Similar-size cities: {peer_h:.2f}', color='#4F1C59', fontsize=9)
ax.set_title('Housing court cases brought by Gateway Cities, FY2018–2024\n (per 10,000 residents per year)',
             color='#4F1C59', fontsize=13)
ax.set_xlabel('Housing court cases per 10,000 residents per year (avg FY2018–2024)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUT}/gateway_housing_court_by_city.png', dpi=200)
plt.show()

### Chart takeaway
- 8 of 26 Gateway Cities bring housing court cases at high rates; Worcester (~23 per 10k/yr) and Springfield (~15) lead.
- 18 of 26 are close to similar-size cities (0.10).
- Most high-rate cities are in central/western MA → possible regional practice in code enforcement (question for GBH, not a conclusion).
- Case type: 98.6% "Housing Court Civil"; city is plaintiff in 99.3%.

  ## 6. Export for Looker
  Small summary only (26 Gateway Cities + similar-size average). No case details.

In [ ]:
  looker = top_housing.rename('housing_cases_per_10k_per_year').reset_index()
  looker['group'] = np.where(looker.housing_cases_per_10k_per_year >= 1,
                             'Gateway City (1+ per 10k)', 'Gateway City (under 1)')
  looker = pd.concat([looker, pd.DataFrame([{
      'municipality': 'Similar-size cities (avg)',
      'housing_cases_per_10k_per_year': round(peer_h, 2),
      'group': 'Comparison'}])], ignore_index=True)

  looker.to_csv(f'{OUT}/looker_gateway_housing_court_2018_2024.csv', index=False)
  print(looker.shape)
  looker.tail(3)